In [ ]:
pip install tweepy

In [ ]:
pip install nltk

In [ ]:
pip install sklearn

In [ ]:
pip install pandas

In [ ]:
pip install re

In [ ]:
pip install matplotlib

In [ ]:
pip install seaborn

In [ ]:
import tweepy
import pandas as pd

api_key = 'p6Fg6WjZeOIQN1FvrsnvCfcbo'
api_key_secret = '7sgTYd66QCOoaEDuOXSoH5TkoTJrzmWvjEDnVdIaO7tqZgeGhg'
access_token = '132425188-3pEgCTglpzvYC605OMINgGkn3fO5pTFMv2NfWY61'
access_token_secret = 'Qkzccwwbf6J4Mrpkem4LHVefHxBT13E3C28x0F8gufIyc'

auth = tweepy.OAuthHandler(api_key, api_key_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth)

def fetch_tweets(keyword, num_tweets):
    tweets = tweepy.Cursor(api.search_tweets, q=keyword, lang="en").items(num_tweets)
    tweet_list = [[tweet.text] for tweet in tweets]
    return pd.DataFrame(tweet_list, columns=["Tweet"])

data = fetch_tweets("women safety OR harassment OR #metoo", 1000)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub(r'http\S+|@\S+|[^A-Za-z\s]', '', text)
    text = text.lower()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text

data['Cleaned_Tweet'] = data['Tweet'].apply(preprocess_text)

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)

    if score['compound'] >= 0.05:
        return 'positive'
    elif score['compound'] <= -0.05:
        return 'negative'
    else:
        return 'neutral'

data['Sentiment'] = data['Cleaned_Tweet'].apply(get_sentiment)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(data['Cleaned_Tweet'])
y = data['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVC(kernel='linear')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='Sentiment', data=data)
plt.title("Sentiment Distribution in Women's Safety Tweets")
plt.show()